# **Hoja de trabajo #2 - Aprendizaje por Refuerzo**

- Paula Barillas - 22764
- Gerardo Pineda - 22880
- Monica Salvatierra - 22249
- Bianca Calderón - 22272

Link del repositorio: https://github.com/paulabaal12/HDT2-RL 

---

## **Task 1**

---

Responda las siguientes preguntas de forma clara, concisa y argumentada. No basta con definir, se espera que usted conecte el concepto con sus implicaciones prácticas o estratégicas.

**1. Elijan un sistema real que pueda modelarse como un k-armed bandit. Su modelado debe incluir:**

a). Definición formal de los 𝑘 brazos: qué representa cada acción y por qué el conjunto de
acciones es finito y discreto en este dominio.

b). Diseño justificado de la función de recompensa: qué se mide, cómo se mide, y qué
supuestos distribucionales son razonables para ese dominio. Si la recompensa no es
gaussiana, arguméntenlo.

c). Análisis de estacionariedad: ¿los valores verdaderos 𝑞∗(𝑎) cambian con el tiempo en este sistema? ¿En qué escala temporal? ¿Qué implicaciones tiene eso para el algoritmo de
actualización?

d). Identificación de restricciones de exploración: ¿hay costos económicos, de seguridad o
regulatorios que limiten cuánto puede explorar el sistema?

---

**Sistema propuesto: Precio dinámico en máquinas expendedoras inteligentes**

Cada vez más máquinas expendedoras tienen pantallas digitales y conectividad, lo que permite cambiar el precio de un producto en tiempo real (no como las de antes, con precio fijo impreso). Es un ejemplo real usado por empresas como Coca-Cola y algunas startups de "smart vending".



**a) Definición formal de los k brazos**

Cada brazo `a` representa un precio distinto al que se puede ofrecer un producto específico (ej. una lata de soda) en un momento dado. Por ejemplo, con k = 5:

a₁: Q. 12.00
a₂: Q. 15.00
a₃: Q. 18.00
a₄: Q. 20.00
a₅: Q. 25.00

El conjunto de acciones es finito y discreto porque:

1. La pantalla/sistema de cobro solo admite ciertos incrementos de precio (normalmente múltiplos de Q 0.50 o Q. 1.00, no cualquier valor continuo).

2. Cambiar de precio tiene que pasar por aprobación del operador de la máquina (o de la marca dueña del producto), quienes típicamente autorizan un catálogo cerrado de precios "permitidos" por razones de imagen de marca y percepción del consumidor.

3. Psicológicamente los precios funcionan por anclas discretas (terminaciones en 0, 5, 9), así que en la práctica el negocio nunca probaría un continuo de precios sino un conjunto de opciones "razonables".

**b) Diseño de la función de recompensa**

- Qué se mide: ingreso neto por transacción potencial (ventas × precio), no solo unidades vendidas.

- Cómo se mide: el sistema de pago de la máquina registra cada intento de compra. Si el cliente completó la compra al precio mostrado, o si "abandonó" (miró el precio y no compró). Con eso se calcula:

`R(a) = precio(a)` si hubo compra, 0 si no hubo compra


Supuestos distribucionales:

En este caso, la recompensa no es gaussiana, esto se debe a que cada evento (una persona parada frente a la máquina) es una decisión binaria. Ya sea compra o no compra a ese precio. Eso es un ensayo Bernoulli con probabilidad p(a) que depende del precio.

La recompensa observada entonces es 0 o precio(a), una distribución discreta de dos puntos, no continua ni simétrica, así que claramente no es gaussiana.

Además, se espera que p(a) sea decreciente en el precio (curva de demanda), lo que hace que el valor esperado q*(a) = precio(a)·p(a) tenga una forma tipo joroba (sube y luego baja) en vez de ser monótono.

**c) Análisis de estacionariedad**

Los valores verdaderos q*(a) sí cambian con el tiempo, en varias escalas:

- Diaria/a cada hora: En oficinas o universidades, la disposición a pagar cambia entre mañana (gente con prisa, más inelástica) y tarde (gente más sensible al precio).

- Semanal: Fin de quincena/quincena de pago vs. días previos a la nómina. La elasticidad de precio de los clientes cambia según cuánto efectivo/saldo disponible tengan.

- Estacional: En época de calor sube la disposición a pagar por una bebida fría (demanda más inelástica); en época fría baja.

- Competencia externa: si abre una tienda de conveniencia cerca, o cambia el precio en la tienda de al lado, la elasticidad de la máquina cambia de forma persistente.

**Implicación para el algoritmo:** Suponer un promedio muestral simple (con peso 1/n) no es adecuado aquí, ya que le resta importancia a las observaciones más recientes. Por eso conviene:

- Usar actualización con paso constante α para dar más peso a transacciones recientes.
- Usar un bandit contextual, donde el contexto es hora del día / día de la semana / temporada, en lugar de tratar el problema como un solo bandit estacionario.
- Considerar métodos con detección de cambio para reaccionar rápido si aparece competencia nueva.

**d) Restricciones de exploración**

- **Económicas directas:** Explorar un precio muy bajo (a₁) implica perder margen real en cada venta durante la exploración. Por otro lado, explorar uno muy alto (a₅) implica perder ventas (y quizás enojar clientes) mientras se recopila información.

- **Regulatorias:** La marca dueña del producto (ej. Coca-Cola) suele fijar un rango de precio "sugerido" o incluso un precio máximo permitido contractualmente, limitando el conjunto de brazos que se pueden probar libremente.

- **Reputacionales:** Si los clientes notan que el precio "cambia todo el tiempo" o que es más caro en ciertos horarios, puede dañar la confianza y reducir ventas futuras de forma persistente. Un costo de exploración que no se recupera fácilmente aunque luego se baje el precio de nuevo.

- **Volumen de datos:** Una sola máquina puede tener pocas transacciones al día (quizás 20-50), así que el "presupuesto de exploración" es limitado. No se puede alcanzar significancia estadística rápido, lo que justifica utilizar máquinas múltiples (varias ubicaciones) como réplicas del mismo experimento para acelerar el aprendizaje.